In [10]:
%load_ext autoreload
%autoreload 2
import math
import sys
sys.path.append('../')

from process_fragment_miner import (
    ProcessFragmentMiner
)

from process_fragment_miner.adapters import load_event_log

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
# 1. Load event log from .xes file (using PM4Py)
event_log = load_event_log("../data/raw/BPIC12.xes.gz")

parsing log, completed traces ::   0%|          | 0/13087 [00:00<?, ?it/s]

In [12]:
# 2. Initialize the miner
miner = ProcessFragmentMiner(
    event_log=event_log,
    scorer="heuristic",
    # scorer_kwargs={"remove_loops": True}
)

# 3. Extract top subtraces using DFS
subtraces = miner.extract_subtraces(max_depth=1000, min_depth=2, top_k=math.inf)

# 4. Select the best disjoint subset of fragments
score, best_traces, scores, method_used = miner.mine_best_fragments(
    subtraces=subtraces,
    score_agg="sum",       # or "mean", "log_likelihood"
    alpha=0.0,             # reward per unique activity
    beam_size=50,
    max_memory_mb=1024,
    method="auto",         # try "dp", "beam", or "auto"
    return_details=True,
    ensure_coverage=True
)

# 5. Output results
print("Best Score: ", score)
print("Best Fragments: ")
display(best_traces)

display(scores)

print("Method used: ", method_used)


Best Score:  3.250743367137457
Best Fragments: 


[['A_PARTLYSUBMITTED', 'A_DECLINED'],
 ['O_CREATED', 'O_SENT', 'W_Nabellen offertes', 'O_SELECTED'],
 ['W_Valideren aanvraag', 'W_Wijzigen contractgegevens'],
 ['A_ACTIVATED', 'W_Nabellen incomplete dossiers', 'A_APPROVED'],
 ['O_ACCEPTED',
  'W_Completeren aanvraag',
  'A_PREACCEPTED',
  'A_ACCEPTED',
  'O_SENT_BACK',
  'A_CANCELLED',
  'A_FINALIZED',
  'A_REGISTERED',
  'W_Beoordelen fraude',
  'W_Afhandelen leads',
  'O_CANCELLED',
  'O_DECLINED',
  'A_SUBMITTED']]

[0.9997084548104956,
 0.9984070463657619,
 0.2857142857142857,
 0.9669135802469135]

Method used:  dp
